In [1]:
import Gmsh: gmsh
using Gridap, GridapGmsh

In [2]:
function create_square_model(h)
    gmsh.initialize()
    gmsh.model.add("unit_square");
    gmsh.model.geo.addPoint(0.0, 0.0, 0.0, h, 1); # last argument is optional identifier, unique per dimension
    gmsh.model.geo.addPoint(1.0, 0.0, 0.0, h, 2);
    gmsh.model.geo.addPoint(1.0, 1.0, 0.0, h, 3);
    gmsh.model.geo.addPoint(0.0, 1.0, 0.0, h, 4);
    gmsh.model.geo.addLine(1, 2, 1); 
    gmsh.model.geo.addLine(2, 3, 2); # line 2 goes from point 2 to point 3
    gmsh.model.geo.addLine(3, 4, 3);
    gmsh.model.geo.addLine(4, 1, 4);
    
    gmsh.model.geo.addCurveLoop([1, 2, 3, 4], 1);

    gmsh.model.geo.addPlaneSurface([1], 1);

    gmsh.model.geo.synchronize();

    # Define physical groups without the string argument
    edges_tag = gmsh.model.addPhysicalGroup(1, [1, 2, 3, 4])   # edges
    corners_tag = gmsh.model.addPhysicalGroup(0, [1, 2, 3, 4]) # corners
    domain_tag = gmsh.model.addPhysicalGroup(2, [1])           # surface
    
    # Set names for the physical groups
    gmsh.model.setPhysicalName(1, edges_tag, "boundary")
    gmsh.model.setPhysicalName(0, corners_tag, "boundary")
    gmsh.model.setPhysicalName(2, domain_tag, "domain")

    gmsh.model.mesh.generate(2); # dimension is 2
    
    model = GmshDiscreteModel(gmsh);
    gmsh.finalize();
    return model
end


create_square_model (generic function with 1 method)

In [3]:
h = 0.02; # target mesh size
model = create_square_model(h) # fix above function above using the tutorial
order = 1
reffe = ReferenceFE(lagrangian, Float64, order)
V = TestFESpace(model, reffe, conformity=:H1, dirichlet_tags="boundary")
U = TrialFESpace(V, 0.0)


Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.00147746s, CPU 0.000811s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.111664s, CPU 0.105812s)
Info    : 3013 nodes 6028 elements


TrialFESpace()

In [4]:
Ω = Triangulation(model)
dΩ = Measure(Ω, 2 * order)

GenericMeasure()

In [5]:
n=6
Δt, T = 2.0^(-n), 2*π # time step, final time
num_steps = Int(round(T / Δt))

402

In [6]:
f(t, x) = 0
f(t) = x -> f(t,x)

f (generic function with 2 methods)

In [7]:
u_exact(t, x) = 0.3*sin(π*x[1])* sin(2*π*x[2]) * cos(π*sqrt(5)*t) + 0.5*sin(2*π*x[1])* sin(2*π*x[2]) * cos(π*sqrt(8)*t) + 0.5*sin(4*π*x[1])* sin(3*π*x[2]) * cos(π*5*t)

u₀ = x -> 0.3*sin(π*x[1])* sin(2*π*x[2]) + 0.5*sin(2*π*x[1])* sin(2*π*x[2])+ 0.5*sin(4*π*x[1])* sin(3*π*x[2])
u₁ = x -> u_exact(Δt,x)
#uₜ_0 = 0

#5 (generic function with 1 method)

In [8]:
m(u, v) = ∫(u*v)dΩ
k(u, v) = ∫(∇(u) ⋅ ∇(v))dΩ

M = assemble_matrix(m, U, V)
K = assemble_matrix(k, U, V)

# LHS matrix for (13.5):
A = (1/Δt^2)*M + (1
    /4)*K

2813×2813 SparseArrays.SparseMatrixCSC{Float64, Int64} with 19293 stored entries:
⎡⠕⣥⣀⠀⠀⡠⠀⠀⢀⡀⣀⡄⡀⣠⠀⠀⠄⡀⠄⠀⢀⠀⠷⣝⡓⠿⠿⠷⠧⡅⣏⡿⡆⣆⠄⠖⠐⠿⠿⠿⎤
⎢⠀⠘⢻⣶⡀⣝⠁⠰⣸⠃⢇⠀⢣⢾⠀⠄⠘⠫⠂⠁⠠⠀⢀⡄⣀⡄⡍⠲⣸⡚⠿⢿⠦⣫⣒⠚⡪⡁⠀⠀⎥
⎢⠀⡠⣄⢬⠿⣧⣠⢀⠸⠂⠙⡂⠐⢠⠀⡀⠀⠀⢀⠀⠐⠀⢠⠈⢜⠠⣜⣕⢮⣿⣗⣶⣻⣮⣧⣠⡼⠇⠀⠀⎥
⎢⠀⠀⢁⡀⠀⢚⣻⣾⢘⡁⡏⡚⣓⣒⡃⡊⠈⡐⠒⠀⠀⠀⠸⠯⠮⡓⣛⢚⣩⣐⣟⣆⢚⠻⡛⣬⡧⠤⠀⠀⎥
⎢⠀⠰⠶⠚⠲⠂⠖⠰⠻⣦⡆⠟⡷⢾⠂⠐⠐⠪⠀⠀⠒⠆⠀⠚⣋⠂⢶⠳⢷⣿⣳⣾⠿⣿⡃⡘⣖⠂⠀⠀⎥
⎢⠀⠼⠉⠑⠳⠠⣫⠩⣬⠍⣿⣿⣿⡿⣤⣧⠈⠴⡴⠀⠈⡇⠠⢃⡸⡤⢬⢾⣽⣟⢿⣽⣿⣿⣿⡧⡭⡁⠀⠀⎥
⎢⠀⣨⣩⣖⠐⣀⢹⢸⣹⣏⣿⡿⣿⣿⣿⣷⣾⣿⣗⠀⣠⡶⢘⠝⡟⢼⣾⣼⢟⣯⣿⣿⣿⣿⣟⣷⣷⡷⠒⠀⎥
⎢⠀⠀⠀⠄⠀⠠⡩⠨⢈⠀⠤⣿⢿⣿⣿⣿⣿⣷⣿⣿⢿⣟⣀⡑⠰⣊⣷⣿⣡⡝⣾⣿⣿⣵⣿⣟⢽⣛⣛⠀⎥
⎢⠀⠡⡶⡀⠀⠀⢂⠠⡰⡀⢂⡄⣾⣿⢿⣿⣿⣿⣷⣸⣿⣮⣽⡿⣳⠷⢿⢿⢿⣮⢷⣭⣫⣚⣽⣫⣺⡍⠭⠉⎥
⎢⠀⠁⠌⠀⠀⠐⠘⠀⠀⠀⠐⠋⠙⠙⣿⣿⣙⣻⢻⣶⣵⢛⡭⣆⢥⡏⣗⣔⡯⠺⣿⡓⣽⡟⣿⣛⠻⣿⣌⡡⎥
⎢⠀⠐⠀⠂⠐⠀⠀⠀⠸⠄⠦⠤⢠⡾⣿⢷⡻⣿⣵⢛⢛⣴⡟⡽⡹⠝⡿⣽⣚⡻⣟⡿⢿⣿⣿⣯⣿⣯⣉⣂⎥
⎢⣝⢧⠀⠴⡀⠒⡶⡆⣠⠀⠤⢂⣖⠔⢄⠸⣷⡿⠣⢯⣟⡭⢿⣷⣿⠖⣽⣵⣷⣎⡼⢾⣎⠷⣮⣿⣆⣿⢹⡏⎥
⎢⣽⡌⠀⠼⠒⡑⢮⠣⠫⠘⠒⡮⣛⣍⡰⢢⢽⡞⡥⠷⣗⠎⢻⠟⢟⢕⣴⣾⠇⢿⠟⣞⣷⡟⢟⣾⣮⣿⢿⣵⎥
⎢⢿⡇⢣⡉⢖⢽⣻⢘⢼⡓⣢⣗⣚⣿⣽⣿⣿⣗⢙⢽⣟⣯⢗⣿⣰⣿⡿⣯⣘⣷⡟⣶⣽⢵⣻⣯⣮⠿⣿⣯⎥
⎢⠍⠧⣲⠺⣮⣷⢃⢺⣽⣷⣷⢿⡿⣵⣅⠾⡻⣷⣫⡋⣾⡸⡹⢿⣭⣅⢶⣼⣿⣿⣿⣿⢲⣺⣖⣳⣺⡗⠟⠙⎥
⎢⣯⡽⣿⣇⢹⣽⠻⢽⣹⣾⣟⣷⣿⣿⣾⣿⡝⣷⢿⠻⣿⡽⣲⣏⣻⢥⢻⣭⣿⣿⣵⣿⣿⣿⣿⣿⣮⣥⢬⡯⎥
⎢⠨⢭⡬⣣⡻⣾⣾⡐⣿⣧⣿⣿⣿⣿⢟⣿⣫⢺⣷⠿⣿⣷⢮⡝⣽⠿⢗⣟⣸⣲⣿⣿⣿⣿⣿⣾⡸⣮⠸⠇⎥
⎢⢠⠅⣸⠘⠉⣻⡛⣬⣉⠨⠿⡿⢿⣽⣿⢿⡷⣻⣿⢻⡿⣿⣮⣿⣻⣵⡿⣾⢼⣹⣿⣿⣻⣿⣿⣿⣦⣅⠥⡆⎥
⎢⣴⡄⠎⠪⠶⠏⠉⡏⠸⠙⠇⠫⢽⡿⣷⢳⡞⠾⣿⣦⡿⣿⣬⣽⣮⣿⣮⡟⢾⠾⠎⣿⡲⣮⠌⢿⢿⣷⠄⣀⎥
⎣⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⠀⠛⠘⡇⠃⠆⡹⠣⢸⡷⠶⢟⣷⡿⣿⣟⠁⡦⡷⠶⠆⠡⠧⠀⢡⣿⣿⎦

In [9]:
uh0 = interpolate_everywhere(u₀, U)   # U^0 
uh1 = interpolate_everywhere(u₁, U) # U^1

SingleFieldFEFunction():
 num_cells: 5824
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 11913426188752217161

In [10]:
if !isdir("tmp") mkdir("tmp") end

createpvd("wave") do pvd
    pvd[0.0] = createvtk(Ω, "tmp/wave4_0.vtu",
                         cellfields=["uh" => uh0, "e" => x -> 0.0])
    pvd[Δt]  = createvtk(Ω, "tmp/wave4_1.vtu",
                         cellfields=["uh" => uh1, "e" => (x -> u_exact(Δt,x)) - uh1])

    # dof-vectors
    uvec_prev = get_free_dof_values(uh0)  # U^{n-1}
    uvec_curr = get_free_dof_values(uh1)  # U^{n}

    e = nothing
    el2 = nothing

    for n in 1:(num_steps-1)
        t_n = n*Δt

        b(v) = ∫( f(t_n)*v )dΩ
        Fn = assemble_vector(b, V)

        # RHS
        rhs = Fn +
              ((2/Δt^2)*M - (1/2)*K) * uvec_curr +
              (-(1/Δt^2)*M - (1/4)*K) * uvec_prev

        # Solve for next step
        uh_next = FEFunction(U, A \ rhs) # solve for next value

        # error
        t_next = (n+1)*Δt
        uex = x -> u_exact(t_next, x)
        e = uex - uh_next

        vtufilename = "tmp/wave4_$(n+1).vtu"
        pvd[t_next] = createvtk(Ω, vtufilename, cellfields=["uh" => uh_next, "e" => e])

        #update
        uvec_prev = uvec_curr
        uvec_curr = A \ rhs
    end

    # last error
    el2 = sqrt(sum( ∫(e*e)*dΩ ))
    println("L2 error norm: $el2")
end

L2 error norm: 0.04399204006245561


404-element Vector{String}:
 "wave.pvd"
 "tmp/wave4_0.vtu"
 "tmp/wave4_1.vtu"
 "tmp/wave4_2.vtu"
 "tmp/wave4_3.vtu"
 "tmp/wave4_4.vtu"
 "tmp/wave4_5.vtu"
 "tmp/wave4_6.vtu"
 "tmp/wave4_7.vtu"
 "tmp/wave4_8.vtu"
 "tmp/wave4_9.vtu"
 "tmp/wave4_10.vtu"
 "tmp/wave4_11.vtu"
 ⋮
 "tmp/wave4_391.vtu"
 "tmp/wave4_392.vtu"
 "tmp/wave4_393.vtu"
 "tmp/wave4_394.vtu"
 "tmp/wave4_395.vtu"
 "tmp/wave4_396.vtu"
 "tmp/wave4_397.vtu"
 "tmp/wave4_398.vtu"
 "tmp/wave4_399.vtu"
 "tmp/wave4_400.vtu"
 "tmp/wave4_401.vtu"
 "tmp/wave4_402.vtu"